# Discovery + Silver: `billing.products`

Caso conocido: `active` llega como texto `True`/`False` (estilo Python), hay que castear a booleano real.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.billing__products", engine)
df.shape

(200, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

product_id               object
sku                      object
name                     object
category                 object
monthly_price            object
active                   object
_source_file             object
_ingested_at     datetime64[ns]
_dag_run_id              object
dtype: object


,product_id,sku,name,category,monthly_price,active,_source_file,_ingested_at,_dag_run_id
0,PRD-00001,SKU-00001,Product 00001,basic,18.99,True,billing/products.csv,2026-07-15 22:29:47.772207,manual__2026-07-15T22:29:45+00:00
1,PRD-00002,SKU-00002,Product 00002,premium,42.16,True,billing/products.csv,2026-07-15 22:29:47.772207,manual__2026-07-15T22:29:45+00:00
2,PRD-00003,SKU-00003,Product 00003,standard,87.4,True,billing/products.csv,2026-07-15 22:29:47.772207,manual__2026-07-15T22:29:45+00:00
3,PRD-00004,SKU-00004,Product 00004,standard,59.01,False,billing/products.csv,2026-07-15 22:29:47.772207,manual__2026-07-15T22:29:45+00:00
4,PRD-00005,SKU-00005,Product 00005,standard,27.98,True,billing/products.csv,2026-07-15 22:29:47.772207,manual__2026-07-15T22:29:45+00:00


## 2. Nulos, duplicados y valores

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("product_id duplicados:", df["product_id"].duplicated().sum())
print()
print("Valores distintos de active (texto crudo):", df["active"].unique())
print("Valores distintos de category:", df["category"].unique())

Nulos por columna:
product_id       0
sku              0
name             0
category         0
monthly_price    0
active           0
_source_file     0
_ingested_at     0
_dag_run_id      0
dtype: int64

product_id duplicados: 0

Valores distintos de active (texto crudo): ['True' 'False']
Valores distintos de category: ['basic' 'premium' 'standard' 'enterprise']


## 3. `monthly_price`: rango de valores

In [4]:
price = pd.to_numeric(df["monthly_price"], errors="coerce")
print("monthly_price no numerico:", price.isna().sum())
print("monthly_price <= 0:", (price <= 0).sum())
print(price.describe())

monthly_price no numerico: 0
monthly_price <= 0: 0
count    200.000000
mean      46.246500
std       49.196794
min        5.000000
25%       18.940000
50%       31.665000
75%       56.625000
max      491.470000
Name: monthly_price, dtype: float64


## 4. Conclusion: reglas de limpieza

Tabla limpia (sin nulos, sin duplicados, precios positivos). Una regla real de tipado:

- `active` -> texto `"True"`/`"False"` a booleano real (`bool`), no un simple `strip()`.
- `sku`, `name` -> `strip()`.
- `category` -> `strip()` + minusculas.
- `monthly_price` -> castear a numerico.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["product_id", "sku", "name", "category", "monthly_price", "active"]].copy()

df_silver["sku"] = df_silver["sku"].str.strip()
df_silver["name"] = df_silver["name"].str.strip()
df_silver["category"] = df_silver["category"].str.strip().str.lower()
df_silver["monthly_price"] = pd.to_numeric(df_silver["monthly_price"], errors="raise")
df_silver["active"] = df_silver["active"].str.strip().str.lower().map({"true": True, "false": False})

df_silver.head()

,product_id,sku,name,category,monthly_price,active
0,PRD-00001,SKU-00001,Product 00001,basic,18.99,True
1,PRD-00002,SKU-00002,Product 00002,premium,42.16,True
2,PRD-00003,SKU-00003,Product 00003,standard,87.40,True
3,PRD-00004,SKU-00004,Product 00004,standard,59.01,False
4,PRD-00005,SKU-00005,Product 00005,standard,27.98,True


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["product_id"].is_unique
assert df_silver["active"].isna().sum() == 0, "algun valor de active no mapeo a booleano"
assert df_silver["monthly_price"].gt(0).all()
print("OK:", len(df_silver), "filas listas para silver")
print(df_silver["active"].dtype)

OK: 200 filas listas para silver
bool


## 7. Escribir en `silver.billing__products`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "billing__products",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.billing__products")

Escrito en silver.billing__products


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.billing__products LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(*) FILTER (WHERE active) AS activos FROM silver.billing__products", engine))
check

   filas  activos
0    200      170


,product_id,sku,name,category,monthly_price,active,_silver_loaded_at
0,PRD-00001,SKU-00001,Product 00001,basic,18.99,True,2026-07-16 19:06:21.510706+00:00
1,PRD-00002,SKU-00002,Product 00002,premium,42.16,True,2026-07-16 19:06:21.510706+00:00
2,PRD-00003,SKU-00003,Product 00003,standard,87.40,True,2026-07-16 19:06:21.510706+00:00
3,PRD-00004,SKU-00004,Product 00004,standard,59.01,False,2026-07-16 19:06:21.510706+00:00
4,PRD-00005,SKU-00005,Product 00005,standard,27.98,True,2026-07-16 19:06:21.510706+00:00
